# Diabetes 30-Day Readmission — Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

pd.set_option('display.max_columns', None)

In [2]:
# '?' is used throughout this dataset to indicate missing values
df = pd.read_csv('diabetic_data.csv', na_values=['?'])

print(f'Raw shape: {df.shape}')
df['readmitted'].value_counts()

Raw shape: (101766, 50)


/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_26565/738007965.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('diabetic_data.csv', na_values=['?'])


readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [3]:
# Remove deceased and hospice patients since they cannot be readmitted
deceased_codes = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(deceased_codes)]
print(f'After removing deceased/hospice: {df.shape}')

# Keep one encounter per patient to avoid data leakage across train/test split
df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')
print(f'After deduplicating patients: {df.shape}')

# Drop administrative IDs, not useful as features
df = df.drop(columns=['encounter_id', 'patient_nbr'])

After removing deceased/hospice: (99343, 50)
After deduplicating patients: (69990, 50)


## Target Variable

In [4]:
# Binarize target: 1 = readmitted within 30 days, 0 = everything else
df['readmitted'] = (df['readmitted'] == '<30').astype(int)

print('Target distribution:')
print(df['readmitted'].value_counts())
print(f'\nMinority class ratio: {df["readmitted"].mean():.3f}')

Target distribution:
readmitted
0    63705
1     6285
Name: count, dtype: int64

Minority class ratio: 0.090


## Drop High-Missingness Columns

In [5]:
# weight is 97% missing, payer_code 40%, medical_specialty 49% — not usable
# diag_1/2/3 are raw ICD-9 codes with ~900 unique values, too granular to use directly
HIGH_MISS_COLS = ['weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']
df = df.drop(columns=HIGH_MISS_COLS)
print(f'Shape after dropping high-missingness columns: {df.shape}')

Shape after dropping high-missingness columns: (69990, 42)


## Feature Definitions

In [6]:
CONTINUOUS_COLS = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses',
]

# Medication dosage columns with values: No, Steady, Up, Down
MEDICATION_COLS = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
    'tolazamide', 'examide', 'citoglipton', 'insulin',
    'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone',
]

# These are stored as integers but represent categories, not ordinal values
ID_CATEGORICAL_COLS = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
]

OTHER_CATEGORICAL_COLS = [
    'race', 'gender', 'age',
    'max_glu_serum', 'A1Cresult',
    'change', 'diabetesMed',
]

print('Continuous:         ', CONTINUOUS_COLS)
print('Medication columns: ', len(MEDICATION_COLS))
print('ID categoricals:    ', ID_CATEGORICAL_COLS)
print('Other categoricals: ', OTHER_CATEGORICAL_COLS)

Continuous:          ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Medication columns:  23
ID categoricals:     ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
Other categoricals:  ['race', 'gender', 'age', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']


## Handle Missing Values

In [7]:
# Mean imputation for continuous columns
for col in CONTINUOUS_COLS:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mean())

# For categorical columns, NaN becomes its own category rather than being dropped
# 'None' for lab results means the test was not performed, which is clinically meaningful
df['race'] = df['race'].fillna('Unknown')
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
df['A1Cresult'] = df['A1Cresult'].fillna('None')

# Drop the two rows with invalid gender entries
df = df[df['gender'] != 'Unknown/Invalid']

print('Remaining NaNs per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Remaining NaNs per column:
Series([], dtype: int64)


## Encode Categorical Features

In [8]:
# Age: ordinal encoding preserving the natural decade order
AGE_ORDER = [
    '[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
    '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)',
]
df['age'] = df['age'].map({bracket: i for i, bracket in enumerate(AGE_ORDER)})

# Medication columns: ordinal encoding (No=0, Steady=1, Up=2, Down=3)
MED_MAP = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
for col in MEDICATION_COLS:
    df[col] = df[col].map(MED_MAP).fillna(0).astype(int)

# Binary columns
df['change']      = (df['change'] == 'Ch').astype(int)
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype(int)

# One-hot encode remaining categoricals, drop_first to avoid multicollinearity
OHE_COLS = (
    ['race', 'gender', 'max_glu_serum', 'A1Cresult']
    + [str(c) for c in ID_CATEGORICAL_COLS]
)
df = pd.get_dummies(df, columns=OHE_COLS, drop_first=True)

print(f'Shape after encoding: {df.shape}')
df.head(2)

Shape after encoding: (69987, 90)


,age,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_Unknown,gender_Male,max_glu_serum_>300,max_glu_serum_None,max_glu_serum_Norm,A1Cresult_>8,A1Cresult_None,A1Cresult_Norm,admission_type_id_2,admission_type_id_3,admission_type_id_4,admission_type_id_5,admission_type_id_6,admission_type_id_7,admission_type_id_8,discharge_disposition_id_2,discharge_disposition_id_3,discharge_disposition_id_4,discharge_disposition_id_5,discharge_disposition_id_6,discharge_disposition_id_7,discharge_disposition_id_8,discharge_disposition_id_9,discharge_disposition_id_10,discharge_disposition_id_12,discharge_disposition_id_15,discharge_disposition_id_16,discharge_disposition_id_17,discharge_disposition_id_18,discharge_disposition_id_22,discharge_disposition_id_23,discharge_disposition_id_24,discharge_disposition_id_25,discharge_disposition_id_27,discharge_disposition_id_28,admission_source_id_2,admission_source_id_3,admission_source_id_4,admission_source_id_5,admission_source_id_6,admission_source_id_7,admission_source_id_8,admission_source_id_9,admission_source_id_10,admission_source_id_11,admission_source_id_13,admission_source_id_14,admission_source_id_17,admission_source_id_20,admission_source_id_22,admission_source_id_25
8,8,13,68,2,28,0,0,0,8,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,False,True,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
9,9,12,33,3,18,0,0,0,8,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,False,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


## Train / Test Split (80 / 20)

In [9]:
X = df.drop(columns=['readmitted'])
y = df['readmitted']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,  # preserve class ratio in both splits
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test  positive rate: {y_test.mean():.3f}')

Train: (55989, 89)  |  Test: (13998, 89)
Train positive rate: 0.090
Test  positive rate: 0.090


## Standardize Continuous Variables

In [10]:
# Fit scaler on training data only, then apply to both splits to prevent data leakage
scaler = StandardScaler()

X_train[CONTINUOUS_COLS] = scaler.fit_transform(X_train[CONTINUOUS_COLS])
X_test[CONTINUOUS_COLS]  = scaler.transform(X_test[CONTINUOUS_COLS])

print('Continuous column means after scaling (train, should be ~0):')
print(X_train[CONTINUOUS_COLS].mean().round(4))

Continuous column means after scaling (train, should be ~0):
time_in_hospital      0.0
num_lab_procedures   -0.0
num_procedures       -0.0
num_medications      -0.0
number_outpatient    -0.0
number_emergency     -0.0
number_inpatient     -0.0
number_diagnoses     -0.0
dtype: float64


## SMOTE (Oversample Minority Class)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

X_train_smote = pd.DataFrame(X_res, columns=X_train.columns)
y_train_smote = pd.Series(y_res, name='readmitted')

print('Before:', y_train.value_counts().to_dict())
print('After: ', y_train_smote.value_counts().to_dict())

X_train_smote.to_csv('X_train_smote.csv', index=False)
y_train_smote.to_csv('y_train_smote.csv', index=False)
print('Saved: X_train_smote.csv, y_train_smote.csv')

## Class Weights (for use in model training)

In [11]:
# Compute class weights from training labels only
# The minority class (readmitted) gets a higher weight to compensate for imbalance
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
CLASS_WEIGHT_DICT = dict(zip(classes, weights))

print('Class weight dict:', CLASS_WEIGHT_DICT)
print(f'\nMinority (readmitted<30) weight: {CLASS_WEIGHT_DICT[1]:.3f}')
print(f'Majority (not readmitted) weight: {CLASS_WEIGHT_DICT[0]:.3f}')

Class weight dict: {np.int64(0): np.float64(0.5493318419968211), np.int64(1): np.float64(5.56772076372315)}

Minority (readmitted<30) weight: 5.568
Majority (not readmitted) weight: 0.549


## Save Preprocessed Splits

In [12]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',  index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv',  index=False)

print('Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv')
print(f'\nFinal feature count: {X_train.shape[1]}')
print(f'Train samples: {X_train.shape[0]}')
print(f'Test  samples: {X_test.shape[0]}')

Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv

Final feature count: 89
Train samples: 55989
Test  samples: 13998
